# 2D Poisson equation with Dirichlet boundary conditions



$$
\frac {\partial^2 u}{\partial x^2} + \frac {\partial^2 u}{\partial y^2}=p(x,y)
$$

$$
p(x,y) = 10(x-1) \cos(5y) - 25(x-1)(y-1) \sin(5y)
$$


$$
x,y \in [0,1]
$$

Boundary Condition
$$
u(x,0)=u(x,1)=u(1,y)=0 \\
u(0,y)=(1-y) \sin(5y)
$$

Exact Solution  
$$
u(x,y) = (1-x)(1-y) \sin(5y)
$$

In [ ]:
pip install -r requirements.txt

In [ ]:
#Import Library
import pandas as pd
import numpy as np

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import time
import shutil

import warnings
warnings.filterwarnings("ignore")

# Define PDE Problem and Initial/Boundary Conditions

In [ ]:
pi = tf.constant(np.pi, dtype=tf.float32)

# PDE Definition
def f(x, y):
    """
    Right-hand side function f(x, y)
    """
    return 10*(x-1)*tf.cos(5*y)-25*(x-1)*(y-1)*tf.sin(5*y)

# Boundary Conditions (Dirichlet)
def f0(y):
    """
    Boundary condition at x = 0
    """
    return (1-y)*tf.sin(5*y)


def f1(y):
    """
    Boundary condition at x = 1
    """
    return tf.zeros_like(y, dtype=tf.float32)

# Initial Conditions
def g0(x):
    """
    Boundary condition at y = 0
    """
    return tf.zeros_like(x, dtype=tf.float32)


def g1(x):
    """
    Boundary condition at y=1
    """
    return tf.zeros_like(x, dtype=tf.float32)

# Data Prep

In [ ]:
n = 200
x = np.linspace(0,1,n)
y = np.linspace(0,1,n)

xx, yy = np.meshgrid(x, y)

n_data = np.size(xx)

xt_point = np.column_stack((xx.ravel(), yy.ravel()))
u_true = np.zeros((n_data, 1))

# Split Data
X_train, X_test, u_train, u_test = train_test_split(xt_point, u_true, test_size=0.2, random_state=42)


# Function

In [ ]:
# Convert array to tensorflow
def tensor(a):
  return tf.convert_to_tensor(a, dtype=tf.float32)

# Initialize Weights and Biases
def initialize_weights(n_layers, seed=None):
    initializer = tf.keras.initializers.GlorotUniform()
    
    params = []
    for in_dim, out_dim in zip(n_layers[:-1], n_layers[1:]):
        W = initializer(shape=(in_dim, out_dim)).numpy()
        b = tf.zeros(shape=(out_dim, 1)).numpy()
        params.append((W, b))
        
    return params

# Build ANN Model
def build_model(n_layers, params):
    model = Sequential()

    for i, ((W, b), units) in enumerate(zip(params, n_layers[1:])):
        activation = 'tanh' if i < len(n_layers) - 2 else 'linear'
        
        layer_kwargs = {
            "units": units,
            "activation": activation,
            "kernel_initializer": tf.keras.initializers.Constant(W),
            "bias_initializer": tf.keras.initializers.Constant(b.flatten())
        }

        # input layer
        if i == 0:
            layer_kwargs["input_shape"] = (n_layers[0],)

        model.add(Dense(**layer_kwargs))
    
    return model

# Build SVD Model
def svd(M, tsvd, r=None):
    U, S, V = np.linalg.svd(M, full_matrices=False)

    if tsvd == 1:  # Thin SVD
        rank_full = len(S)

        if r is not None:
            print(f"[INFO] 'rank' parameter is ignored for Thin SVD (type 1) and rank={rank_full} will be used.")
            
        S = np.diag(S)
        return U, S, V

    elif tsvd == 2:  # Compact SVD
        if r is None:
            raise ValueError("For Compact SVD (type 2), var 'rank' must be provided.")

        S = S[S >= r]
        o = S.shape[0]
        U = U[:, :o]
        S = np.diag(S)
        V = V[:o, :]
        return U, S, V

    elif tsvd == 3:  # Truncated SVD
        if r is None:
            raise ValueError("For Truncated SVD (type 3), var 'rank' must be provided.")

        U = U[:, :r]
        S = np.diag(S[:r])
        V = V[:r, :]
        return U, S, V

    else:
        raise ValueError(
            f"Invalid type_svd={tsvd}. "
            "Allowed values: 1 (Thin), 2 (Compact), 3 (Truncated)."
        )
  
class SVDLayer2(tf.keras.layers.Layer):
    def __init__(self, weight, type_svd=1, rank=None, activation=None, **kwargs):
        super(SVDLayer2, self).__init__(**kwargs)
        self.weight_init = weight
        self.type_svd = type_svd
        self.rank = rank
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        Ur, Sr, Vr = svd(self.weight_init, self.type_svd, r=self.rank)

        self.U = self.add_weight(
            name="U",
            shape=Ur.shape,
            initializer=tf.constant_initializer(Ur),
            trainable=True
        )

        self.S = self.add_weight(
            name="S",
            shape=Sr.shape,
            initializer=tf.constant_initializer(Sr),
            trainable=True
        )

        self.V = self.add_weight(
            name="V",
            shape=Vr.shape,
            initializer=tf.constant_initializer(Vr),
            trainable=True
        )

    def call(self, inputs):
        result = tf.matmul(inputs, self.U)
        result = tf.matmul(result, self.S)
        result = tf.matmul(result, self.V)

        if self.activation is not None:
            result = self.activation(result)

        return result

    def get_config(self):
        config = super().get_config()
        config.update({
            "type_svd": self.type_svd,
            "rank": self.rank,
            "activation": tf.keras.activations.serialize(self.activation),
        })
        return config

def build_svd_model(n_layers, params, type_svd=1, rank=None):
    weights = [p[0] for p in params]
    biases  = [p[1] for p in params]

    model = Sequential()

    # Input layer
    model.add(Dense(
        units=n_layers[1],
        activation='tanh',
        kernel_initializer=tf.keras.initializers.Constant(weights[0]),
        bias_initializer=tf.keras.initializers.Constant(biases[0].flatten()),
        input_shape=(n_layers[0],)
    ))

    # SVD layer
    model.add(SVDLayer2(
        weight=weights[1],
        type_svd=type_svd,
        rank=rank,
        activation='tanh'
    ))

    # Output layer
    model.add(Dense(
        units=n_layers[-1],
        activation='linear',
        kernel_initializer=tf.keras.initializers.Constant(weights[2]),
        bias_initializer=tf.keras.initializers.Constant(biases[2].flatten())
    ))

    return model

# Get Model
def get_model(model_type, n_layers, params, type_svd=None, rank=None):

    if model_type == "ann":
        return build_model(n_layers, params)

    elif model_type == "ann_svd":
        if type_svd is None:
            type_svd = 1 

        return build_svd_model(
            n_layers=n_layers,
            params=params,
            type_svd=type_svd,
            rank=rank
        )

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

# Trial solution
def A(x,y):
  a = tf.constant(0, dtype=tf.float32)
  b = tf.constant(1, dtype=tf.float32)
  return (1-x)*f0(y)+x*f1(y)+(1-y)*(g0(x)-((1-x)*g0(a)+x*g0(b)))+y*(g1(x)-((1-x)*g1(a)+x*g1(b)))

def UA(x,y,N):
  return A(x,y) + x * (1-x) * y * (1-y) * N

# Loss Function
def _loss(model, var_input):
    x_a = tensor(var_input[:, [0]])
    y_a = tensor(var_input[:, [1]])

    with tf.GradientTape(persistent=True) as tape:
        tape.watch([x_a, y_a])

        input = tf.concat([x_a, y_a], axis=1)

        # Forward pass
        N = model(input)
        u = UA(x_a, y_a, N)

        # First derivatives
        u_x = tape.gradient(u, x_a)
        u_y = tape.gradient(u, y_a)

    # Second derivatives
    u_xx = tape.gradient(u_x, x_a)
    u_yy = tape.gradient(u_y, y_a)

    del tape

    # PDE residual
    residual = u_yy + u_xx - f(x_a, y_a)

    # Loss = MSE residual
    loss = tf.reduce_mean(tf.square(residual))

    return loss

# Training Loop
def train_ann(model, var_input, n_epoch, lr=0.001):
    optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=lr)

    start_time = time.time() 

    loss_history = []

    for epoch in range(1, n_epoch + 1):
        with tf.GradientTape() as tape:
            loss = _loss(model, var_input)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        loss_value = loss.numpy()
        loss_history.append(loss_value)

        elapsed = time.time() - start_time

        print(f"[Epoch {epoch:4d}/{n_epoch}] "
              f"Loss: {loss_value:.6e} | "
              f"Time: {elapsed:.2f}s")

    total_time = time.time() - start_time
    avg_time = total_time / n_epoch

    print(f"\n Total Training Time: {total_time:.2f} seconds")
    print(f" Average Time per Epoch: {avg_time:.2f} seconds")

    return model, loss_history, total_time

# Plot Loss History
def plot_loss_history(loss_history):
    plt.figure(figsize=(5, 5))
    plt.plot(loss_history, label='Training Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss History')
    plt.legend()
    plt.show()

# Approximate Solution
def appx_sltn(model, var_input):
    x_a = tensor(var_input[:, [0]])
    y_a = tensor(var_input[:, [1]])

    input = tf.concat([x_a, y_a], axis=1)

    N = model(input)
    u_approx = UA(x_a, y_a, N)

    return u_approx.numpy()

# Grid Plot of Approximate Solution
def fig_grid(output, title):
    fig = plt.figure(figsize=(8, 8))

    output = output.reshape((n, n))

    fig.suptitle(title, fontsize=14, y=0.7)

    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(xx, yy, output, cmap='viridis')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')

    ax2 = fig.add_subplot(122)
    surf = ax2.imshow(output, cmap='viridis', origin='lower', extent=[0, 1, 0, 1])
    fig.colorbar(surf, shrink=0.6, aspect=5.35)
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')

    fig.subplots_adjust(wspace=0.4, top=0.88)

    plt.show()

# Evaluate Model
def evaluate_model(model, X_train, X_test):
    train_loss = _loss(model, X_train).numpy()
    test_loss = _loss(model, X_test).numpy()
    L_inf = np.max(np.abs(appx_sltn(model, X_test) - U(X_test[:, 0:1], X_test[:, 1:2])))
    L_rms = np.sqrt(np.mean((appx_sltn(model, X_test) - U(X_test[:, 0:1], X_test[:, 1:2]))**2))

    print(f"Train Loss: {train_loss:.6e}")
    print(f"Test Loss: {test_loss:.6e}")
    print(f"L_inf Error: {L_inf:.6e}")
    print(f"L_rms Error: {L_rms:.6e}")

# Save Model
def save_model(model, name):
  model.save(f'{name}')

  folder_path = f'{name}'
  zip_path = f'{name}'
  shutil.make_archive(zip_path, 'zip', folder_path)

In [ ]:
def main(n_layers, n_epoch,
         model_type="ann",
         type_svd=None,
         rank=None,
         exact_solution=None,
         save_path=None):

    # Init weights
    params = initialize_weights(n_layers)

    # Build model
    try:
        model = get_model(
            model_type=model_type,
            n_layers=n_layers,
            params=params,
            type_svd=type_svd,
            rank=rank
        )
    except ValueError as e:
        print(f"[ERROR] {e}")
        return None, None, None

    # Train
    model, loss_history, total_time = train_ann(model, X_train, n_epoch)

    # Visualization
    plot_loss_history(loss_history)

    Ua = appx_sltn(model, xt_point)

    if exact_solution is not None:
        fig_grid(Ua, "Approximate Solution")
        fig_grid(abs(Ua - exact_solution), "Absolute Error")
    else:
        fig_grid(Ua, "Approximate Solution")

    # Evaluation
    evaluate_model(model, X_train, X_test)

    # Save Model
    if save_path is not None:
        svd_name = {1: "thin", 2: "compact", 3: "truncated"}.get(type_svd, "unknown")
        suffix = f"_{svd_name}" if model_type == "ann_svd" else ""
        filename = f"{save_path}_{model_type}{suffix}"

        save_model(model, filename)
        print(f"\n [INFO] Model saved as: {filename}")

    return model, loss_history, total_time

# Plot Analytic Solution

In [ ]:
def U(x, y):
    result = (1-x)*(1-y)*np.sin(5*y)
    return result

x = xt_point[:, [0]]
y = xt_point[:, [1]]

Ue = U(x, y)

fig_grid(Ue, 'Approximate Solution')


# Training ANN

In [ ]:
n_layers = [2, 50, 70, 1]   #[input, hidden1, hidden2, output]
n_epoch = 100
name_path = "(p4)"

model_ann, hloss_ann, train_time_ann = main(n_layers, 
                                            n_epoch, 
                                            model_type="ann", 
                                            exact_solution=Ue, 
                                            save_path=name_path)

# ANN-SVD

## ANN-Thin SVD

In [ ]:
model_ann_thin, hloss_ann_thin, train_time_ann_thin = main(n_layers, 
                                                           n_epoch, 
                                                           model_type="ann_svd", 
                                                           type_svd=1, 
                                                           exact_solution=Ue,
                                                           save_path=name_path)

## ANN-Compact SVD

In [ ]:
model_ann_compact, hloss_ann_compact, train_time_ann_compact = main(n_layers, 
                                                                    n_epoch, 
                                                                    model_type="ann_svd", 
                                                                    type_svd=2, 
                                                                    rank=0.5, 
                                                                    exact_solution=Ue,
                                                                    save_path=name_path)

## ANN-Truncated SVD

In [ ]:
model_ann_truncated, hloss_ann_truncated, train_time_ann_truncated = main(n_layers, 
                                                                          n_epoch, 
                                                                          model_type="ann_svd", 
                                                                          type_svd=3, 
                                                                          rank=3, 
                                                                          exact_solution=Ue,
                                                                          save_path=name_path)